In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrwellsdavid/unsw-nb15/NUSW-NB15_features.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_1.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_2.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_3.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_4.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_LIST_EVENTS.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_testing-set.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv


In [2]:
# ============================================
# NetSentinel - Step 1: Environment and Dataset
# ============================================

import os
import sys
import platform

print("=" * 60)
print("NETSENTINEL: ENVIRONMENT VERIFICATION")
print("=" * 60)

print("Python version:", sys.version)
print("Operating system:", platform.platform())

print("\n" + "=" * 60)
print("DATASET FILES")
print("=" * 60)

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(f"File: {file}")
        print(f"Path: {file_path}")
        print(f"Size: {size_mb:.2f} MB")
        print("-" * 40)

print("\n" + "=" * 60)
print("AVAILABLE RESOURCES")
print("=" * 60)

print("CPU cores:", os.cpu_count())

try:
    import psutil

    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"System RAM: {ram_gb:.2f} GB")
except ImportError:
    print("RAM information unavailable.")

try:
    import tensorflow as tf

    print("TensorFlow version:", tf.__version__)

    gpus = tf.config.list_physical_devices("GPU")
    print("Available GPUs:", len(gpus))

    for gpu in gpus:
        print("GPU device:", gpu.name)

except ImportError:
    print("TensorFlow is not installed.")

NETSENTINEL: ENVIRONMENT VERIFICATION
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Operating system: Linux-6.18.55+-x86_64-with-glibc2.39

DATASET FILES
File: NUSW-NB15_features.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/NUSW-NB15_features.csv
Size: 0.00 MB
----------------------------------------
File: UNSW-NB15_1.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_1.csv
Size: 161.15 MB
----------------------------------------
File: UNSW-NB15_2.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_2.csv
Size: 157.57 MB
----------------------------------------
File: UNSW-NB15_3.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_3.csv
Size: 147.43 MB
----------------------------------------
File: UNSW-NB15_4.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_4.csv
Size: 93.07 MB
----------------------------------------
File: UNSW-NB15_LIST_EVENTS.csv
Path: /kaggle/input/datasets/mrwellsdavid/unsw-

2026-10-29 04:35:30.200641: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (82332, 45)


In [6]:
print(df.columns.tolist())

['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


In [7]:
print(df["label"].value_counts())

label
1    45332
0    37000
Name: count, dtype: int64


In [8]:
print(df["attack_cat"].value_counts())

attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


In [9]:
print(df.isnull().sum().sum())

0


In [10]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [11]:
print(df.dtypes)

id                     int64
dur                  float64
proto                 object
service               object
state                 object
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [12]:
categorical_features = ["proto", "service", "state"]

print("Categorical input features:")
for feature in categorical_features:
    print("-", feature)

Categorical input features:
- proto
- service
- state


In [13]:
for column in ["proto", "service", "state"]:
    print(f"\n{column}")
    print("Number of unique values:", df[column].nunique())
    print(df[column].unique())


proto
Number of unique values: 131
['udp' 'arp' 'tcp' 'igmp' 'ospf' 'sctp' 'gre' 'ggp' 'ip' 'ipnip' 'st2'
 'argus' 'chaos' 'egp' 'emcon' 'nvp' 'pup' 'xnet' 'mux' 'dcn' 'hmp' 'prm'
 'trunk-1' 'trunk-2' 'xns-idp' 'leaf-1' 'leaf-2' 'irtp' 'rdp' 'netblt'
 'mfe-nsp' 'merit-inp' '3pc' 'idpr' 'ddp' 'idpr-cmtp' 'tp++' 'ipv6' 'sdrp'
 'ipv6-frag' 'ipv6-route' 'idrp' 'mhrp' 'i-nlsp' 'rvd' 'mobile' 'narp'
 'skip' 'tlsp' 'ipv6-no' 'any' 'ipv6-opts' 'cftp' 'sat-expak' 'ippc'
 'kryptolan' 'sat-mon' 'cpnx' 'wsn' 'pvp' 'br-sat-mon' 'sun-nd' 'wb-mon'
 'vmtp' 'ttp' 'vines' 'nsfnet-igp' 'dgp' 'eigrp' 'tcf' 'sprite-rpc' 'larp'
 'mtp' 'ax.25' 'ipip' 'aes-sp3-d' 'micp' 'encap' 'pri-enc' 'gmtp' 'ifmp'
 'pnni' 'qnx' 'scps' 'cbt' 'bbn-rcc' 'igp' 'bna' 'swipe' 'visa' 'ipcv'
 'cphb' 'iso-tp4' 'wb-expak' 'sep' 'secure-vmtp' 'xtp' 'il' 'rsvp' 'unas'
 'fc' 'iso-ip' 'etherip' 'pim' 'aris' 'a/n' 'ipcomp' 'snp' 'compaq-peer'
 'ipx-n-ip' 'pgm' 'vrrp' 'l2tp' 'zero' 'ddx' 'iatp' 'stp' 'srp' 'uti' 'sm'
 'smp' 'isis' 'ptp'

In [14]:
print(pd.crosstab(df["label"], df["attack_cat"]))

attack_cat  Analysis  Backdoor   DoS  Exploits  Fuzzers  Generic  Normal  \
label                                                                      
0                  0         0     0         0        0        0   37000   
1                677       583  4089     11132     6062    18871       0   

attack_cat  Reconnaissance  Shellcode  Worms  
label                                         
0                        0          0      0  
1                     3496        378     44  


In [15]:
X = df.drop(columns=["label", "attack_cat"])
y = df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (82332, 43)
y shape: (82332,)

Target distribution:
label
1    45332
0    37000
Name: count, dtype: int64


In [16]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:")
print(categorical_cols)

print("\nNumber of categorical features:", len(categorical_cols))

print("\nNumber of numerical features:", len(numerical_cols))


Categorical features:
['proto', 'service', 'state']

Number of categorical features: 3

Number of numerical features: 40


In [17]:
print(X[numerical_cols].describe().T)

                     count          mean           std   min           25%  \
id                 82332.0  4.116650e+04  2.376735e+04   1.0  20583.750000   
dur                82332.0  1.006756e+00  4.710444e+00   0.0      0.000008   
spkts              82332.0  1.866647e+01  1.339164e+02   1.0      2.000000   
dpkts              82332.0  1.754594e+01  1.155741e+02   0.0      0.000000   
sbytes             82332.0  7.993908e+03  1.716423e+05  24.0    114.000000   
dbytes             82332.0  1.323379e+04  1.514715e+05   0.0      0.000000   
rate               82332.0  8.241089e+04  1.486204e+05   0.0     28.606114   
sttl               82332.0  1.809677e+02  1.015134e+02   0.0     62.000000   
dttl               82332.0  9.571300e+01  1.166677e+02   0.0      0.000000   
sload              82332.0  6.454902e+07  1.798618e+08   0.0  11202.466797   
dload              82332.0  6.305470e+05  2.393001e+06   0.0      0.000000   
sloss              82332.0  4.753692e+00  6.464962e+01   0.0    

In [18]:
X = X.drop(columns=["id"])

print("Updated X shape:", X.shape)
print("Remaining features:", X.shape[1])

Updated X shape: (82332, 42)
Remaining features: 42


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

Training set: (65865, 42)
Validation set: (16467, 42)

Training target distribution:
label
1    36265
0    29600
Name: count, dtype: int64

Validation target distribution:
label
1    9067
0    7400
Name: count, dtype: int64


In [20]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_val_cat = encoder.transform(X_val[categorical_cols])

print("Training categorical shape:", X_train_cat.shape)
print("Validation categorical shape:", X_val_cat.shape)

Training categorical shape: (65865, 151)
Validation categorical shape: (16467, 151)


In [25]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train_num = scaler.fit_transform(X_train[numerical_cols])
# X_val_num = scaler.transform(X_val[numerical_cols])

# print("Training numerical shape:", X_train_num.shape)
# print("Validation numerical shape:", X_val_num.shape)

KeyError: "['id'] not in index"

In [22]:
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of numerical features:", len(numerical_cols))
print(numerical_cols)


Number of numerical features: 39
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_val_num = scaler.transform(X_val[numerical_cols])

print("Training numerical shape:", X_train_num.shape)
print("Validation numerical shape:", X_val_num.shape)

Training numerical shape: (65865, 39)
Validation numerical shape: (16467, 39)


In [24]:
import numpy as np

X_train_processed = np.hstack([X_train_num, X_train_cat])
X_val_processed = np.hstack([X_val_num, X_val_cat])

print("Final training shape:", X_train_processed.shape)
print("Final validation shape:", X_val_processed.shape)

Final training shape: (65865, 190)
Final validation shape: (16467, 190)


In [29]:
print("NaN values:", np.isnan(X_train_processed).sum())
print("Infinite values:", np.isinf(X_train_processed).sum())

print("\nValidation NaN values:", np.isnan(X_val_processed).sum())
print("Validation Infinite values:", np.isinf(X_val_processed).sum())

NaN values: 0
Infinite values: 0

Validation NaN values: 0
Validation Infinite values: 0


In [28]:
import tensorflow as tf

# Convert features to float32
X_train_processed = X_train_processed.astype("float32")
X_val_processed = X_val_processed.astype("float32")

# Convert labels to float32
y_train = y_train.to_numpy().astype("float32")
y_val = y_val.to_numpy().astype("float32")

print("X_train:", X_train_processed.shape, X_train_processed.dtype)
print("X_val:  ", X_val_processed.shape, X_val_processed.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)

AttributeError: 'numpy.ndarray' object has no attribute 'to_numpy'

In [30]:
# Reshape for CNN/RNN/LSTM branches
X_train_seq = X_train_processed.reshape(-1, 190, 1)
X_val_seq = X_val_processed.reshape(-1, 190, 1)

print("Training sequence shape:", X_train_seq.shape)
print("Validation sequence shape:", X_val_seq.shape)

Training sequence shape: (65865, 190, 1)
Validation sequence shape: (16467, 190, 1)


In [31]:
from tensorflow.keras import layers, Model

# Input: 190 processed features
ann_input = layers.Input(shape=(190,), name="ann_input")

# ANN branch
ann = layers.Dense(128, activation="relu")(ann_input)
ann = layers.Dropout(0.3)(ann)

ann = layers.Dense(64, activation="relu")(ann)
ann = layers.Dropout(0.3)(ann)

ann_output = layers.Dense(32, activation="relu")(ann)

# Create ANN branch model
ann_model = Model(
    inputs=ann_input,
    outputs=ann_output,
    name="ANN_Branch"
)

ann_model.summary()

Model: "ANN_Branch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ann_input (InputLayer)          │ (None, 190)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        24,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,784 (135.88 KB)

 Trainable params: 34,784 (135.88 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
# CNN branch
cnn_input = layers.Input(shape=(190, 1), name="cnn_input")

cnn = layers.Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu",
    padding="same"
)(cnn_input)

cnn = layers.MaxPooling1D(pool_size=2)(cnn)

cnn = layers.Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu",
    padding="same"
)(cnn)

cnn = layers.GlobalAveragePooling1D()(cnn)

cnn_output = layers.Dense(32, activation="relu")(cnn)

# Create CNN branch model
cnn_model = Model(
    inputs=cnn_input,
    outputs=cnn_output,
    name="CNN_Branch"
)

cnn_model.summary()

Model: "CNN_Branch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ cnn_input (InputLayer)          │ (None, 190, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 190, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 95, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 95, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,488 (29.25 KB)

 Trainable params: 7,488 (29.25 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# LSTM branch
lstm_input = layers.Input(shape=(190, 1), name="lstm_input")

lstm = layers.LSTM(64, return_sequences=True)(lstm_input)
lstm = layers.Dropout(0.3)(lstm)

lstm = layers.LSTM(32)(lstm)
lstm = layers.Dropout(0.3)(lstm)

lstm_output = layers.Dense(32, activation="relu")(lstm)

# Create LSTM branch model
lstm_model = Model(
    inputs=lstm_input,
    outputs=lstm_output,
    name="LSTM_Branch"
)

lstm_model.summary()

Model: "LSTM_Branch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_input (InputLayer)         │ (None, 190, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 190, 64)        │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 190, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,368 (118.62 KB)

 Trainable params: 30,368 (118.62 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
# RNN branch
rnn_input = layers.Input(shape=(190, 1), name="rnn_input")

rnn = layers.SimpleRNN(64, return_sequences=True)(rnn_input)
rnn = layers.Dropout(0.3)(rnn)

rnn = layers.SimpleRNN(32)(rnn)
rnn = layers.Dropout(0.3)(rnn)

rnn_output = layers.Dense(32, activation="relu")(rnn)

# Create RNN branch model
rnn_model = Model(
    inputs=rnn_input,
    outputs=rnn_output,
    name="RNN_Branch"
)

rnn_model.summary()

Model: "RNN_Branch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rnn_input (InputLayer)          │ (None, 190, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 190, 64)        │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 190, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,384 (32.75 KB)

 Trainable params: 8,384 (32.75 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# Fuse ANN, CNN, LSTM and RNN outputs
fused_features = layers.Concatenate(name="feature_fusion")([
    ann_model.output,
    cnn_model.output,
    lstm_model.output,
    rnn_model.output
])

print("Fused feature shape:", fused_features.shape)

Fused feature shape: (None, 128)


In [36]:
# Reshape fused features for attention
attention_input = layers.Reshape((128, 1), name="attention_reshape")(
    fused_features
)

# Self-attention
attention_output = layers.MultiHeadAttention(
    num_heads=4,
    key_dim=16,
    name="feature_attention"
)(
    attention_input,
    attention_input
)

# Residual connection + normalization
attention_output = layers.Add()([
    attention_input,
    attention_output
])

attention_output = layers.LayerNormalization()(
    attention_output
)

print("Attention output shape:", attention_output.shape)

Attention output shape: (None, 128, 1)


In [37]:
# Classification head
attention_pooled = layers.GlobalAveragePooling1D(
    name="attention_pooling"
)(attention_output)

classifier = layers.Dense(
    64,
    activation="relu",
    name="classifier_dense"
)(attention_pooled)

classifier = layers.Dropout(
    0.3,
    name="classifier_dropout"
)(classifier)

final_output = layers.Dense(
    1,
    activation="sigmoid",
    name="attack_probability"
)(classifier)

# Complete NetSentinel hybrid model
netsentinel_model = Model(
    inputs=[
        ann_input,
        cnn_input,
        lstm_input,
        rnn_input
    ],
    outputs=final_output,
    name="NetSentinel_Hybrid_Attention"
)

netsentinel_model.summary()


Model: "NetSentinel_Hybrid_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ann_input           │ (None, 190)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_input           │ (None, 190, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_input          │ (None, 190, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_input           │ (None, 190, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     24,448 │ ann_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 190, 64)   │        256 │ cnn_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 190, 64)   │     16,896 │ lstm_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 190, 64)   │      4,224 │ rnn_input[0][0]   │
│ (SimpleRNN)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 95, 64)    │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 190, 64)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 190, 64)   │          0 │ simple_rnn[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 95, 32)    │      6,176 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_1        │ (None, 32)        │      3,104 │ dropout_4[0][0]   │
│ (SimpleRNN)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ conv1d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ simple_rnn_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0] 

 Total params: 81,668 (319.02 KB)

 Trainable params: 81,668 (319.02 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.optimizers import Adam

netsentinel_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc")
    ]
)

print("NetSentinel compiled successfully!")

NetSentinel compiled successfully!


In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:")
print(class_weight_dict)

Class weights:
{np.float32(0.0): np.float64(1.1125844594594594), np.float32(1.0): np.float64(0.9081069902109472)}


In [39]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPUs: []


In [40]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = netsentinel_model.fit(
    [
        X_train_processed,
        X_train_seq,
        X_train_seq,
        X_train_seq
    ],
    y_train,
    validation_data=(
        [
            X_val_processed,
            X_val_seq,
            X_val_seq,
            X_val_seq
        ],
        y_val
    ),
    epochs=20,
    batch_size=128,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

ValueError: You must call `compile()` before using the model.

In [41]:
# Check whether our preprocessed data is still in memory

required_vars = [
    "X_train_processed",
    "X_val_processed",
    "y_train",
    "y_val"
]

for var in required_vars:
    print(f"{var}: {'Available' if var in globals() else 'Missing'}")

X_train_processed: Available
X_val_processed: Available
y_train: Available
y_val: Available


In [43]:

import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Calculate class weights
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
baseline_class_weights = dict(zip(classes, weights))

# Build baseline ANN
baseline_ann = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
], name="NetSentinel_Baseline_ANN")

# Compile
baseline_ann.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc")
    ]
)

# Stop if validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train
baseline_history = baseline_ann.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=15,
    batch_size=256,
    class_weight=baseline_class_weights,
    callbacks=[early_stopping],
    verbose=1
)

print("\nBaseline ANN training completed.")


Epoch 1/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8486 - loss: 0.2989 - precision: 0.9006 - recall: 0.8149 - roc_auc: 0.9409 - val_accuracy: 0.8982 - val_loss: 0.1961 - val_precision: 0.9748 - val_recall: 0.8368 - val_roc_auc: 0.9784
Epoch 2/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9177 - loss: 0.1786 - precision: 0.9534 - recall: 0.8944 - roc_auc: 0.9792 - val_accuracy: 0.9332 - val_loss: 0.1515 - val_precision: 0.9760 - val_recall: 0.9008 - val_roc_auc: 0.9863
Epoch 3/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9354 - loss: 0.1477 - precision: 0.9622 - recall: 0.9187 - roc_auc: 0.9858 - val_accuracy: 0.9429 - val_loss: 0.1320 - val_precision: 0.9773 - val_recall: 0.9176 - val_roc_auc: 0.9888
Epoch 4/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9432 - loss: 0.1312 - precision: 0.9657 - recall: 0.9299 - roc_auc: 0.9886 - val_accuracy: 0.9484 - val_loss: 0.1239 - val_precision: 0.9592 - val_recall: 0.9466 - val_roc_auc: 0.9896


In [44]:

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Predict attack probabilities and binary labels
y_val_prob = baseline_ann.predict(X_val_processed, verbose=0).ravel()
y_val_pred = (y_val_prob >= 0.5).astype(int)

# Calculate evaluation metrics
print("BASELINE ANN — VALIDATION RESULTS")
print("-" * 40)
print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_val_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Normal", "Attack"],
    digits=4,
    zero_division=0
))


BASELINE ANN — VALIDATION RESULTS
----------------------------------------
Accuracy : 0.9579
Precision: 0.9709
Recall   : 0.9520
F1-score : 0.9614
ROC-AUC  : 0.9931

Confusion Matrix:
[[7141  259]
 [ 435 8632]]

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9426    0.9650    0.9537      7400
      Attack     0.9709    0.9520    0.9614      9067

    accuracy                         0.9579     16467
   macro avg     0.9567    0.9585    0.9575     16467
weighted avg     0.9582    0.9579    0.9579     16467



In [45]:

import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"

for filename in [
    "UNSW_NB15_training-set.csv",
    "UNSW_NB15_testing-set.csv"
]:
    path = os.path.join(DATA_DIR, filename)

    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        print(f"\nFile: {filename}")
        print("Shape:", temp_df.shape)
        print("Columns:", len(temp_df.columns))
        print("Label distribution:")
        print(temp_df["label"].value_counts().sort_index().to_dict())
    else:
        print(f"\nFile not found: {path}")



File: UNSW_NB15_training-set.csv
Shape: (82332, 45)
Columns: 45
Label distribution:
{0: 37000, 1: 45332}

File: UNSW_NB15_testing-set.csv
Shape: (175341, 45)
Columns: 45
Label distribution:
{0: 56000, 1: 119341}


In [46]:

import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"

# Use the files according to their actual row counts, not their filenames
file_a = pd.read_csv(os.path.join(DATA_DIR, "UNSW_NB15_training-set.csv"))
file_b = pd.read_csv(os.path.join(DATA_DIR, "UNSW_NB15_testing-set.csv"))

if len(file_a) == 82332 and len(file_b) == 175341:
    test_df = file_a.copy()
    train_df = file_b.copy()
elif len(file_a) == 175341 and len(file_b) == 82332:
    train_df = file_a.copy()
    test_df = file_b.copy()
else:
    raise ValueError("Unexpected dataset sizes. Stop and inspect the files.")

# Verify feature schemas match
if set(train_df.columns) != set(test_df.columns):
    raise ValueError("Training and test columns do not match.")

print("Training data:", train_df.shape)
print("Held-out test data:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts().sort_index())

print("\nHeld-out test labels:")
print(test_df["label"].value_counts().sort_index())

print("\nColumn schemas match:", set(train_df.columns) == set(test_df.columns))


Training data: (175341, 45)
Held-out test data: (82332, 45)

Training labels:
label
0     56000
1    119341
Name: count, dtype: int64

Held-out test labels:
label
0    37000
1    45332
Name: count, dtype: int64

Column schemas match: True


In [47]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate features and binary target
drop_cols = ["label", "attack_cat"]
X_all = train_df.drop(columns=drop_cols + ["id"], errors="ignore").copy()
y_all = train_df["label"].astype("int32").copy()

X_test_raw = test_df.drop(columns=drop_cols + ["id"], errors="ignore").copy()
y_test = test_df["label"].astype("int32").copy()

# Create training and validation sets from training data only
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# Identify categorical and numerical columns
categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = [
    col for col in X_train_raw.columns if col not in categorical_cols
]

# Encode categorical columns: fit on training data only
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32
)

X_train_cat = encoder.fit_transform(X_train_raw[categorical_cols])
X_val_cat = encoder.transform(X_val_raw[categorical_cols])
X_test_cat = encoder.transform(X_test_raw[categorical_cols])

# Scale numerical columns: fit on training data only
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_raw[numerical_cols])
X_val_num = scaler.transform(X_val_raw[numerical_cols])
X_test_num = scaler.transform(X_test_raw[numerical_cols])

# Combine numerical and categorical features
X_train_processed = np.hstack([X_train_num, X_train_cat]).astype("float32")
X_val_processed = np.hstack([X_val_num, X_val_cat]).astype("float32")
X_test_processed = np.hstack([X_test_num, X_test_cat]).astype("float32")

# Prepare targets
y_train = y_train.to_numpy().astype("float32")
y_val = y_val.to_numpy().astype("float32")
y_test = y_test.to_numpy().astype("float32")

# Validate shapes and numerical values
print("Training:", X_train_processed.shape, y_train.shape)
print("Validation:", X_val_processed.shape, y_val.shape)
print("Held-out test:", X_test_processed.shape, y_test.shape)
print("Categorical features:", categorical_cols)
print("Numerical features:", len(numerical_cols))
print("NaN values:", np.isnan(X_train_processed).sum(),
      np.isnan(X_val_processed).sum(), np.isnan(X_test_processed).sum())
print("Infinite values:", np.isinf(X_train_processed).sum(),
      np.isinf(X_val_processed).sum(), np.isinf(X_test_processed).sum())


Training: (140272, 192) (140272,)
Validation: (35069, 192) (35069,)
Held-out test: (82332, 192) (82332,)
Categorical features: ['proto', 'service', 'state']
Numerical features: 39
NaN values: 0 0 0
Infinite values: 0 0 0


In [48]:

import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(42)

# Balanced class weights from corrected training labels
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
corrected_class_weights = dict(zip(classes, weights))

# Baseline ANN for 192 input features
corrected_baseline = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
], name="NetSentinel_Corrected_Baseline")

corrected_baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc")
    ]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

baseline_history = corrected_baseline.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=15,
    batch_size=256,
    class_weight=corrected_class_weights,
    callbacks=[early_stopping],
    verbose=1
)

print("Corrected baseline ANN training completed.")


Epoch 1/15
548/548 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9079 - loss: 0.2013 - precision: 0.9400 - recall: 0.9235 - roc_auc: 0.9727 - val_accuracy: 0.9200 - val_loss: 0.1488 - val_precision: 0.9727 - val_recall: 0.9080 - val_roc_auc: 0.9866
Epoch 2/15
548/548 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9264 - loss: 0.1455 - precision: 0.9631 - recall: 0.9274 - roc_auc: 0.9855 - val_accuracy: 0.9214 - val_loss: 0.1421 - val_precision: 0.9743 - val_recall: 0.9085 - val_roc_auc: 0.9875
Epoch 3/15
548/548 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9281 - loss: 0.1402 - precision: 0.9662 - recall: 0.9268 - roc_auc: 0.9865 - val_accuracy: 0.9191 - val_loss: 0.1430 - val_precision: 0.9790 - val_recall: 0.9005 - val_roc_auc: 0.9881
Epoch 4/15
548/548 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9293 - loss: 0.1370 - precision: 0.9683 - recall: 0.9264 - roc_auc: 0.9871 - val_accuracy: 0.9229 - val_loss: 0.1391 - val_precision: 0.9794 - val_recall: 0.9057 - val_roc_auc: 0.9886


In [49]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Predict on validation data only
y_val_prob = corrected_baseline.predict(
    X_val_processed, verbose=0
).ravel()

y_val_pred = (y_val_prob >= 0.5).astype(int)

# Calculate evaluation metrics
print("=== NetSentinel: Corrected Baseline ANN ===")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_val, y_val_pred, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, y_val_prob):.4f}")
print(f"PR-AUC:    {average_precision_score(y_val, y_val_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred,
    target_names=["Normal", "Attack"],
    zero_division=0
))

=== NetSentinel: Corrected Baseline ANN ===
Accuracy:  0.9320
Precision: 0.9770
Recall:    0.9219
F1-score:  0.9486
ROC-AUC:   0.9898
PR-AUC:    0.9952

Confusion Matrix:
[[10681   519]
 [ 1864 22005]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.85      0.95      0.90     11200
      Attack       0.98      0.92      0.95     23869

    accuracy                           0.93     35069
   macro avg       0.91      0.94      0.92     35069
weighted avg       0.94      0.93      0.93     35069

